In [5]:
import sys
from pathlib import Path

In [6]:
class ImportMyModules:

    def __init__(self) -> None:
        self.cwd = Path.cwd().resolve()

    def _get_repo_root(self, folder: str) -> Path:
        return next(
            parent for parent in [self.cwd, *self.cwd.parents]
            if (parent / "lib" / "peregrin" / folder).exists()
        )

    def insert_path(self, folder: str) -> None:
        repo_root = self._get_repo_root("src")

        package_root = repo_root / "lib" / "peregrin"
        print(f"Adding {package_root} to sys.path")
        sys.path.insert(0, str(package_root))

importer = ImportMyModules()
importer.insert_path("src")
importer.insert_path("data")

Adding C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin to sys.path
Adding C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin to sys.path


In [7]:
from src.compute.stats import stats
from src.plot.tracks.reconstruct import reconstruct
from data import naive_b

%load_ext autoreload
%autoreload

In [8]:
spots = stats.spots(naive_b.ctr)
tracks = stats.tracks(spots)

In [9]:
spots

,track_id,time_point,x_coordinate,y_coordinate,condition,replicate,track_uid,frame,distance,cum_track_length,...,cum_speed_mean,cum_mean_straight_line_speed,cum_forward_progression_linearity,direction,directional_change,cum_sum_directional_change,cum_mean_directional_change,cum_mean_directional_change_rate,cum_direction_mean,cum_direction_var
track_uid,,,,,,,,,,,,,,,,,,,,,
0,0.0,0.0,619.516680,480.682139,ctr,1,0,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,0.0,60.0,618.630088,479.993816,ctr,1,0,1,1.122423,1.122423,...,0.018707,0.009354,0.500000,-2.481428,NaN,NaN,NaN,NaN,-2.481428,0.000000
0,0.0,120.0,619.329748,480.267255,ctr,1,0,2,0.751194,1.873617,...,0.015613,0.002528,0.161916,0.372564,163.521700,163.521700,163.521700,0.908454,-1.054432,0.856695
0,0.0,180.0,619.194400,480.430855,ctr,1,0,3,0.212330,2.085947,...,0.011589,0.001703,0.146935,2.261967,108.254803,271.776503,135.888251,0.566201,2.331303,0.760171
0,0.0,240.0,618.897852,480.451666,ctr,1,0,4,0.297278,2.383225,...,0.009930,0.002201,0.221667,3.071532,46.384630,318.161132,106.053711,0.353512,2.764630,0.598435
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1025,551.0,3420.0,3.771472,682.606576,ctr,3,1025,57,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1025,551.0,3480.0,10.843262,681.219758,ctr,3,1025,58,7.206488,7.206488,...,0.120108,0.060054,0.500000,-0.193648,NaN,NaN,NaN,NaN,-0.193648,0.000000
1025,551.0,3540.0,13.651912,681.384400,ctr,3,1025,59,2.813471,10.019960,...,0.083500,0.055310,0.662394,0.058553,14.450042,14.450042,14.450042,0.080278,-0.067548,0.007940


In [10]:
reconstruct(spots, tracks)

C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin\src\plot\categorizer.py:72: CategorizerWarning: Conditions not specified. <- Returning all conditions.
  self._checkcats()
C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin\src\plot\categorizer.py:72: CategorizerWarning: Replicates not specified. <- Returning all replicates.
  self._checkcats()
C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin\src\plot\categorizer.py:72: CategorizerWarning: Conditions not specified. <- Returning all conditions.
  self._checkcats()
C:\Users\modri\Desktop\Repositories\peregrin\lib\peregrin\src\plot\categorizer.py:72: CategorizerWarning: Replicates not specified. <- Returning all replicates.
  self._checkcats()


AttributeError: 'ReconstructTracks' object has no attribute 'c_mode'

In [44]:
from typing import Optional
import numpy as np


class ColorGenerator:
    _HEX = np.array([f"{i:02x}" for i in range(256)], dtype="<U2")

    def __init__(self): ...

    def random_color(
        self,
        n: Optional[int] = 1,
        *,
        code: str = "rgba",
        a: float = 1.0,
        **kwargs,
    ) -> np.ndarray:
        n = 1 if n is None else n
        rng = np.random.default_rng(kwargs.get("seed", 42))

        rgb = rng.integers(0, 256, size=(n, 3), dtype=np.uint8)
        return self._color_value(rgb, code=code, a=a)

    def random_grey(
        self,
        n: Optional[int] = 1,
        *,
        code: str = "rgba",
        a: float = 1.0,
        **kwargs,
    ) -> np.ndarray:
        n = 1 if n is None else n
        rng = np.random.default_rng(kwargs.get("seed", 42))

        grey = rng.integers(0, 240, size=(n, 1), dtype=np.uint8)
        rgb = np.repeat(grey, 3, axis=1)
        return self._color_value(rgb, code=code, a=a)

    def _color_value(
        self,
        rgb: np.ndarray,
        *,
        code: str = "rgba",
        a: float = 1.0,
    ) -> np.ndarray:
        rgb = np.asarray(rgb, dtype=np.uint8)

        if rgb.ndim == 1:
            rgb = rgb.reshape(1, -1)

        alpha = float(np.clip(a, 0.0, 1.0))

        match code:
            case "hex":
                alpha_hex = np.full((rgb.shape[0], 1), round(alpha * 255), dtype=np.uint8)
                rgba = np.hstack((rgb, alpha_hex))
                parts = self._HEX[rgba]

                out = np.char.add("#", parts[:, 0])
                out = np.char.add(out, parts[:, 1])
                out = np.char.add(out, parts[:, 2])
                out = np.char.add(out, parts[:, 3])
                return out

            case "rgb":
                return np.array(
                    [f"rgb({r}, {g}, {b})" for r, g, b in rgb],
                    dtype=object,
                )

            case "rgba":
                return np.array(
                    [f"rgba({r}, {g}, {b}, {alpha})" for r, g, b in rgb],
                    dtype=object,
                )

            case _:
                raise ValueError(
                    "Unsupported color code. Use one of: 'hex', 'rgb', 'rgba'."
                )

In [13]:
import pandas as pd

# create a dummy array of shape (1, 100 000) with "row" str values
data = np.asarray(["row"] * 100_000)

df = pd.DataFrame(data, columns=["dummy_column"])
df

,dummy_column
0,row
1,row
2,row
3,row
4,row
...,...
99995,row
99996,row
99997,row
99998,row


In [52]:
random_color = ColorGenerator().random_color

# generate a random color in rgba for each row in the series
df["random_color"] = random_color(n=df.shape[0], code="rgb", a=1.0)

df

,dummy_column,random_color
0,row,"rgb(136, 38, 217)"
1,row,"rgb(22, 205, 251)"
2,row,"rgb(33, 198, 193)"
3,row,"rgb(255, 145, 167)"
4,row,"rgb(97, 86, 90)"
...,...,...
99995,row,"rgb(179, 228, 214)"
99996,row,"rgb(255, 123, 36)"
99997,row,"rgb(70, 1, 136)"
99998,row,"rgb(201, 213, 87)"
